# ☕ Cafe Sales Data Cleaning

**Goal:** Clean a cafe sales dataset containing intentional data quality issues (missing values, invalid placeholders, wrong dtypes) and prepare it for analysis.

**Data source:** [Cafe Sales Dirty Data - Kaggle](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training)

**Methodology:** Instead of filling every missing value the same way, each column is handled based on its nature:
- If there's a logical/mathematical relationship with another column → infer from it
- If there's no reliable relationship → leave the value explicitly missing (Unknown / NaN) instead of fabricating data


## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dirty_cafe_sales.csv')
df.shape

(10000, 8)

### First look at the data

In [2]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


## 2. Diagnosing the Problems

Before any cleaning, we need to understand the scope of the problem. This dataset has a trick: missing values aren't all standard `NaN` - a large chunk are literally the strings `"ERROR"` or `"UNKNOWN"`, which pandas won't recognize as missing automatically.

In [4]:
# Check unique values in each text column - notice ERROR and UNKNOWN
for col in df.select_dtypes(include='object').columns:
    print(col, ':', df[col].unique()[:8])

Transaction ID : ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411'
 'TXN_2602893' 'TXN_4433211' 'TXN_6699534']
Item : ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan]
Quantity : ['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]
Price Per Unit : ['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR']
Total Spent : ['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0']
Payment Method : ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]
Location : ['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']
Transaction Date : ['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28']


In [5]:
# Count of ERROR and UNKNOWN per column
for col in df.columns:
    err = (df[col] == 'ERROR').sum()
    unk = (df[col] == 'UNKNOWN').sum()
    if err or unk:
        print(f"{col}: ERROR={err}, UNKNOWN={unk}")

Item: ERROR=292, UNKNOWN=344
Quantity: ERROR=170, UNKNOWN=171
Price Per Unit: ERROR=190, UNKNOWN=164
Total Spent: ERROR=164, UNKNOWN=165
Payment Method: ERROR=306, UNKNOWN=293
Location: ERROR=358, UNKNOWN=338
Transaction Date: ERROR=142, UNKNOWN=159


**Decision:** Convert `"ERROR"` and `"UNKNOWN"` to real `NaN` across the entire dataframe at once, so we can handle them with standard pandas tools (isna, fillna, dropna) instead of chasing them as text column by column.

In [6]:
df.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

## 3. Item Column (Product Name)

969 missing values (~9.7%). Instead of filling them randomly (e.g. with the mode) or dropping them and losing other useful data in the row, we tried a smarter question:

**Does `Price Per Unit` uniquely identify the product?**

In [7]:
# Convert Price Per Unit to numeric first
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

# For each price, check which products it's associated with (from rows where both are known)
known = df.dropna(subset=['Item', 'Price Per Unit'])
price_item_map = known.groupby('Price Per Unit')['Item'].unique()
for price, items in price_item_map.items():
    print(f"{price}: {list(items)}")

1.0: ['Cookie']
1.5: ['Tea']
2.0: ['Coffee']
3.0: ['Cake', 'Juice']
4.0: ['Smoothie', 'Sandwich']
5.0: ['Salad']


**Result:** 4 out of 6 prices map to exactly one product with full confidence (1.0→Cookie, 1.5→Tea, 2.0→Coffee, 5.0→Salad).
The prices 3.0 and 4.0 are ambiguous between two products each, so price alone can't resolve them.

**Decision:** Use the price-based inference only for the confident cases (the 4 unique prices), and for the rest (unknown price, or one of the ambiguous prices) label it clearly as `"Unknown Item"`.

In [8]:
before_missing = df['Item'].isna().sum()

confident_map = {1.0: 'Cookie', 1.5: 'Tea', 2.0: 'Coffee', 5.0: 'Salad'}
mask = df['Item'].isna() & df['Price Per Unit'].isin(confident_map.keys())
df.loc[mask, 'Item'] = df.loc[mask, 'Price Per Unit'].map(confident_map)

filled_from_price = mask.sum()
df['Item'] = df['Item'].fillna('Unknown Item')

print(f"Originally missing: {before_missing}")
print(f"Filled via price inference: {filled_from_price}")
print(f"Set to Unknown Item: {before_missing - filled_from_price}")
df['Item'].value_counts()

Originally missing: 969
Filled via price inference: 468
Set to Unknown Item: 501


Item
Coffee          1284
Salad           1270
Cookie          1209
Tea             1199
Juice           1171
Cake            1139
Sandwich        1131
Smoothie        1096
Unknown Item     501
Name: count, dtype: int64

## 4. Numeric Columns: Quantity, Price Per Unit, Total Spent

These three columns are linked by a simple equation:

$$Total\ Spent = Quantity \times Price\ Per\ Unit$$

So if only one of the three is missing, we can calculate it from the other two instead of guessing.

In [9]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

missing_count = df[['Quantity','Price Per Unit','Total Spent']].isna().sum(axis=1)
print("All three missing together:", (missing_count == 3).sum())
print("Only one missing (calculable):", (missing_count == 1).sum())
print("Two missing (not directly calculable):", (missing_count == 2).sum())

All three missing together: 0
Only one missing (calculable): 1398
Two missing (not directly calculable): 58


**Before calculating, we verified the data is mathematically reliable** - i.e. in rows where all three were known, does the equation actually hold?

In [10]:
check = df.dropna(subset=['Quantity','Price Per Unit','Total Spent']).copy()
check['expected'] = check['Quantity'] * check['Price Per Unit']
mismatches = (check['Total Spent'] - check['expected']).abs() > 0.01
print(f"Rows with a mismatch: {mismatches.sum()} out of {len(check)}")

Rows with a mismatch: 0 out of 8544


**Zero mismatches** - the data is 100% reliable, so it's safe to use the equation for calculation.

In [11]:
# Calculate the missing column from the other two, when only one is missing
m1 = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[m1, 'Total Spent'] = df.loc[m1, 'Quantity'] * df.loc[m1, 'Price Per Unit']

m2 = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna() & (df['Price Per Unit'] != 0)
df.loc[m2, 'Quantity'] = df.loc[m2, 'Total Spent'] / df.loc[m2, 'Price Per Unit']

m3 = df['Price Per Unit'].isna() & df['Total Spent'].notna() & df['Quantity'].notna() & (df['Quantity'] != 0)
df.loc[m3, 'Price Per Unit'] = df.loc[m3, 'Total Spent'] / df.loc[m3, 'Quantity']

print(f"Total Spent calculated: {m1.sum()} | Quantity calculated: {m2.sum()} | Price calculated: {m3.sum()}")
df[['Quantity','Price Per Unit','Total Spent']].isna().sum()

Total Spent calculated: 462 | Quantity calculated: 441 | Price calculated: 495


Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64

After this step, some rows still have **two columns missing at once**. We noticed many of them have a known `Item`, and since each product's price is nearly fixed (as seen earlier), we can use the **average price per item** as a reliable estimate for the missing price, then calculate the rest.

In [12]:
item_avg_price = df.dropna(subset=['Price Per Unit']).groupby('Item')['Price Per Unit'].mean()
item_avg_price

Item
Cake            3.000000
Coffee          2.000000
Cookie          1.000000
Juice           3.000000
Salad           5.000000
Sandwich        4.000000
Smoothie        4.000000
Tea             1.500000
Unknown Item    3.412121
Name: Price Per Unit, dtype: float64

In [13]:
mask_price = df['Price Per Unit'].isna() & (df['Item'] != 'Unknown Item') & df['Item'].isin(item_avg_price.index)
df.loc[mask_price, 'Price Per Unit'] = df.loc[mask_price, 'Item'].map(item_avg_price)

# Retry the calculation now that we have more prices available
m1 = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[m1, 'Total Spent'] = df.loc[m1, 'Quantity'] * df.loc[m1, 'Price Per Unit']

m2 = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna() & (df['Price Per Unit'] != 0)
df.loc[m2, 'Quantity'] = (df.loc[m2, 'Total Spent'] / df.loc[m2, 'Price Per Unit']).round()

print(f"Price filled from item average: {mask_price.sum()}")
df[['Quantity','Price Per Unit','Total Spent']].isna().sum()

Price filled from item average: 32


Quantity          23
Price Per Unit     6
Total Spent       23
dtype: int64

**Result:** We went from ~500 missing rows per column down to fewer than 25 - over 95% of the problem was resolved through logic, not guessing. The few remaining rows (no known Item and no other usable column) will stay missing on purpose - addressed in the final section.

## 5. Payment Method and Location

The missing rate here is high (32% and 40%), and unlike the previous columns, **there's no other column that can help us infer them**. The existing values are also fairly balanced in frequency (no clearly dominant value).

**Decision:** Filling with the mode (most frequent value) would mean fabricating 32-40% of the data, which would distort any downstream analysis. The correct approach is to leave them explicitly as `"Unknown"`.

In [14]:
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

print(df['Payment Method'].value_counts())
print()
print(df['Location'].value_counts())

Payment Method
Unknown           3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

Location
Unknown     3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64


## 6. Transaction Date

Convert the column to a real `datetime` type. Missing/corrupted values will automatically become `NaT` (the datetime equivalent of NaN). There's no logical relationship that lets us infer a date from another column, so these are left missing.

In [15]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
print("Missing dates:", df['Transaction Date'].isna().sum())
print("Date range:", df['Transaction Date'].min(), "->", df['Transaction Date'].max())

Missing dates: 460
Date range: 2023-01-01 00:00:00 -> 2023-12-31 00:00:00


## 7. Remaining Missing Rows

After all attempts, a small number of rows (fewer than 25 in the numeric columns, and 460 in the date column) don't have enough information to infer from. **Decision: leave them missing (NaN/NaT) rather than dropping the rows or fabricating numbers.**

Reasoning: dropping the rows would lose valid data in other columns (like Item and Payment Method), and fabricating numbers would harm any downstream statistical analysis. Leaving the value explicitly missing is the safest choice, and whoever uses the data afterward can decide for themselves how to handle it (e.g. filter it out at analysis time).

## 8. Final Report

In [16]:
print("Number of rows:", len(df))
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nOverall completeness: {(1 - df.isnull().sum().sum() / df.size) * 100:.1f}%")

Number of rows: 10000

Missing values per column:
Transaction ID        0
Item                  0
Quantity             23
Price Per Unit        6
Total Spent          23
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

Overall completeness: 99.4%


In [17]:
df.dtypes

Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

**The data reached 99.4% completeness, and every value was either filled or left missing based on a documented, reasoned decision - not a blanket rule applied blindly.**

In [18]:
df.to_csv('cafe_sales_cleaned.csv', index=False)
print("Cleaned file saved: cafe_sales_cleaned.csv")

Cleaned file saved: cafe_sales_cleaned.csv
